## Clase 9 - Progresióny audio

-----

### **1. Discusión: Progresión y Feedback al Jugador**



  * **Separar Diseño y Lógica:** La idea central es no tener que cambiar el código para cambiar un nivel. Los niveles son **datos** que el juego carga. Usar archivos de texto (`.txt`,`.csv`, json, xml, etc) es una forma clásica de lograrlo. Cada caracter en el archivo de texto es un código que nuestro programa interpreta para colocar un ladrillo de un tipo específico.
  * **Vidas y Reintentos:** Un juego es más justo y menos frustrante si los errores no son catastróficos. El sistema de **vidas** es un pilar del diseño de juegos. Perder una pelota no termina el juego, solo consume un intento. Esto anima al jugador a arriesgarse y seguir jugando.
  * **UI Centralizada:** La Interfaz de Usuario (puntuación, vidas, nivel) debe ser consistente. En lugar de que cada parte del juego dibuje su propio texto, centralizamos esta lógica en una clase de UI. Esto asegura que si cambiamos la fuente o el color, el cambio se aplica en todo el juego. Nuestro `engine/ui.py` y `engine/dialogue_manager.py` que cumplen esta función.

-----

### **2. Estructura de Archivos Requerida**

Para que esta clase funcione, la estructura de carpetas debe ser la que ya establecimos:

```
/tu_proyecto/
|
|-- engine/
|   |-- ... (todos nuestros módulos del motor)
|
|-- arkanoid/
|   |-- assets/
|   |   |-- sprites/
|   |       |-- arkanoid.json
|   |       |-- blocks_bricks.png
|   |
|   |-- levels/
|   |   |-- level_1.txt
|   |   |-- level_2.txt
|   |
|   |-- scenes/
|   |   |-- menu_scene.py
|   |   |-- game_scene.py
|   |   |-- game_over_scene.py
|   |
|   |-- entities.py
|   |-- level_loader.py
|   |-- game.py
|   |-- main.py
```


#### **`arkanoid/level_loader.py` **

Este módulo se encarga de leer los archivos `.txt`.

```python
# arkanoid/level_loader.py
import pygame
import os
from .entities import Brick

def load_level_from_file(file_path, assets):
    """Carga un nivel desde un archivo de texto y devuelve un grupo de sprites."""
    ladrillos = pygame.sprite.Group()
    
    # Mapeo de caracteres a nombres de sprites de tu arkanoid.json y vidas
    brick_map = {
        'G': {'name': 'bloque_gris', 'health': 1},
        'M': {'name': 'bloque_marron', 'health': 2},
        'A': {'name': 'bloque_azul', 'health': 3},
        # Añade más mapeos aquí. 'S' puede ser cualquier sprite de tu JSON
        'S': {'name': 'sprite_11', 'health': 1},
    }
    
    brick_width, brick_height = 32, 16
    
    with open(file_path, 'r') as f:
        for j, line in enumerate(f):
            for i, char in enumerate(line.strip()):
                if char in brick_map:
                    data = brick_map[char]
                    sprite_name = data['name']
                    health = data['health']
                    
                    try:
                        brick_img = assets.get_sprite('main_ss', sprite_name)
                        x = i * (brick_width + 4) + 120
                        y = j * (brick_height + 4) + 80
                        
                        ladrillo = Brick(x, y, brick_img, assets, health, sprite_name)
                        ladrillos.add(ladrillo)
                    except (ValueError, KeyError) as e:
                        print(f"Error al crear ladrillo desde el nivel: {e}")

    return ladrillos
```

#### **`arkanoid/scenes/game_scene.py` **

Esta es la clase que implementa toda la lógica de vidas, puntuación, niveles y UI.

```python
# arkanoid/scenes/game_scene.py
import pygame
import os
import random
from engine.ui import Scene
from ..entities import Paddle, Ball, Brick, PowerUp
from ..level_loader import load_level_from_file

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        # Fuentes y colores para la UI
        self.font = pygame.font.Font(None, 36)
        self.BLANCO = (255, 255, 255)
        self.AZUL_OSCURO = (13, 20, 56)
        
        self.setup_new_game()

    def setup_new_game(self):
        """Inicializa un juego completamente nuevo (vidas, puntuación, nivel)."""
        self.vidas = 3
        self.puntuacion = 0
        self.nivel_actual = 1
        self.setup_level()

    def setup_level(self):
        """Configura el estado para el nivel actual."""
        self.estado_juego = 'loading'
        self.todos_los_sprites = pygame.sprite.Group()
        self.ladrillos = pygame.sprite.Group()
        self.pelotas = pygame.sprite.Group()
        self.powerups = pygame.sprite.Group()

        try:
            # Creación de la pala y la pelota
            pala_img = self.game.assets.get_sprite('main_ss', 'paleta')
            self.pala = Paddle(self.game.screen.get_width() // 2, self.game.screen.get_height() - 40, pala_img)
            self.todos_los_sprites.add(self.pala)
            
            self.spawn_ball() # Crea la pelota inicial

            # Carga del nivel desde archivo
            level_path = os.path.join(self.game.game_dir, 'levels', f'level_{self.nivel_actual}.txt')
            if not os.path.exists(level_path):
                self.estado_juego = 'victoria_final' # No hay más niveles, juego ganado
                return

            self.ladrillos = load_level_from_file(level_path, self.game.assets)
            self.todos_los_sprites.add(self.ladrillos)
            
            self.estado_juego = 'jugando'

        except (ValueError, KeyError) as e:
            print(f"Error al configurar la escena del juego: {e}")
            self.game.running = False

    def spawn_ball(self):
        """Crea una nueva pelota sobre la pala."""
        try:
            bola_img = self.game.assets.get_sprite('main_ss', 'pelota_bocha')
            bola = Ball(self.pala.rect.centerx, self.pala.rect.top - 20, bola_img)
            self.pelotas.add(bola)
            self.todos_los_sprites.add(bola)
        except (ValueError, KeyError) as e:
            print(f"Error al crear la pelota: {e}")
            self.game.running = False

    def handle_events(self, events):
        for event in events:
            if self.estado_juego in ('game_over', 'victoria_final'):
                if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
                    self.game.change_scene('menu')

    def update(self):
        if self.estado_juego != 'jugando':
            return
        
        self.todos_los_sprites.update(self.game.screen.get_width(), self.game.screen.get_height())
        
        # --- Lógica de Colisiones ---
        for bola in self.pelotas:
            # ... (código de colisión bola-pala y bola-ladrillos sin cambios)...

        # ... (código de colisión pala-powerups sin cambios) ...
        
        # --- Lógica de Vidas y Niveles ---
        for bola in self.pelotas.copy():
            if bola.rect.top > self.game.screen.get_height():
                bola.kill()
        
        if not self.pelotas: # Si no quedan pelotas en pantalla
            self.vidas -= 1
            if self.vidas > 0:
                self.spawn_ball() # Crea una nueva pelota
            else:
                self.estado_juego = 'game_over'
        
        if not self.ladrillos: # Si no quedan ladrillos
            self.nivel_actual += 1
            self.setup_level() # Intenta cargar el siguiente nivel

    def draw(self, surface):
        surface.fill(self.AZUL_OSCURO)
        self.todos_los_sprites.draw(surface)
        
        # --- UI Centralizada ---
        self.draw_ui(surface)
        
        if self.estado_juego == 'game_over':
            self.game.dialogue_manager.show_message("GAME OVER")
            self.game.dialogue_manager.draw()
        elif self.estado_juego == 'victoria_final':
            self.game.dialogue_manager.show_message("¡HAS GANADO!")
            self.game.dialogue_manager.draw()

    def draw_ui(self, surface):
        """Dibuja la información en pantalla (puntuación, vidas, nivel)."""
        score_text = self.font.render(f"Puntuación: {self.puntuacion}", True, self.BLANCO)
        surface.blit(score_text, (10, 10))
        
        lives_text = self.font.render(f"Vidas: {self.vidas}", True, self.BLANCO)
        lives_rect = lives_text.get_rect(right=surface.get_width() - 10, y=10)
        surface.blit(lives_text, lives_rect)

        level_text = self.font.render(f"Nivel: {self.nivel_actual}", True, self.BLANCO)
        level_rect = level_text.get_rect(centerx=surface.get_width() / 2, y=10)
        surface.blit(level_text, level_rect)

```

#### **`arkanoid/game.py` (Actualizado)**

EL gestor principal ahora necesita crear una instancia del `DialogueManager` para que las escenas puedan usarlo.

```python
# arkanoid/game.py
# ...
from engine.dialogue_manager import DialogueManager # Importamos el gestor de diálogo

class ArkanoidGame:
    def __init__(self):
        # ...
        self.assets = AssetManager()
        # Creamos el gestor de diálogo para que esté disponible globalmente
        self.dialogue_manager = DialogueManager(self.screen)
        # ...
```

### **Resumen de la Actualización**

  * **Lógica en `GameScene`:** Toda la lógica del juego, incluyendo la gestión de vidas, puntuación y el avance entre niveles, ahora reside en `arkanoid/scenes/game_scene.py`.
  * **Carga de Niveles:** El nuevo archivo `arkanoid/level_loader.py` se encarga de leer los `.txt` y crear los muros de ladrillos, separando el diseño de la lógica.
  * **Gestión de Vidas:** Cuando te quedas sin pelotas, el juego resta una vida y crea una pelota nueva. Solo si las vidas llegan a cero, el estado cambia a `game_over`.
  * **Progresión de Niveles:** Al destruir todos los ladrillos, el juego incrementa el contador de nivel e intenta cargar `level_2.txt`, `level_3.txt`, etc. Si no encuentra el siguiente archivo, ganas el juego.
  * **UI Centralizada:** La `GameScene` tiene su propio método `draw_ui` para mostrar la información, y utiliza el `DialogueManager` del motor para los mensajes grandes de fin de partida.

### ** El Nuevo Módulo del Motor: `engine/audio.py`**

Primero, se crea este nuevo archivo en la carpeta `engine/`. Este módulo se encargará de toda la gestión del audio.

```python
# engine/audio.py
import pygame

class AudioManager:
    """
    Gestor centralizado para cargar y reproducir música y efectos de sonido.
    """
    def __init__(self, num_channels=16):
        try:
            pygame.mixer.pre_init(44100, -16, 2, 512)
            pygame.mixer.init()
            pygame.mixer.set_num_channels(num_channels)
            self.sounds = {}
            self.music_path = None
            print("AudioManager inicializado correctamente.")
        except pygame.error as e:
            print(f"Error al inicializar AudioManager: {e}. El audio estará desactivado.")
            self.sounds = None # Desactivamos el audio si hay error

    def load_sound(self, name, path):
        """Carga un efecto de sonido y lo asocia con un nombre clave."""
        if self.sounds is None: return
        try:
            self.sounds[name] = pygame.mixer.Sound(path)
        except pygame.error as e:
            print(f"Error al cargar el sonido '{name}' desde '{path}': {e}")

    def play_sound(self, name, loops=0):
        """Reproduce un sonido por su nombre en el primer canal disponible."""
        if self.sounds is None or name not in self.sounds:
            return
        
        # Busca un canal que no esté ocupado para reproducir el sonido
        channel = pygame.mixer.find_channel()
        if channel:
            channel.play(self.sounds[name], loops)

    def load_music(self, path):
        """Carga la ruta para la música de fondo."""
        if self.sounds is None: return
        self.music_path = path

    def play_music(self, loops=-1, volume=0.5):
        """Reproduce la música de fondo."""
        if self.sounds is None or not self.music_path:
            return
        try:
            pygame.mixer.music.load(self.music_path)
            pygame.mixer.music.set_volume(volume)
            pygame.mixer.music.play(loops)
        except pygame.error as e:
            print(f"Error al reproducir la música desde '{self.music_path}': {e}")

    def stop_music(self):
        if self.sounds is None: return
        pygame.mixer.music.stop()
```

-----

### **2. Integración en el Juego Arkanoid**

Ahora, usaremos este nuevo gestor en nuestro juego.

#### **Estructura de Carpetas de Assets**

Para mantener el orden, creamos una subcarpeta para el audio:

```
/arkanoid/
|-- assets/
|   |-- sprites/
|   |-- audio/         <-- ¡NUEVO!
|   |   |-- hit_paddle.wav
|   |   |-- hit_brick.wav
|   |   |-- lose_life.mp3
|   |   |-- music.ogg
```

Podemos descargar de: https://pixabay.com/sound-effects/search/paddle/

#### **`arkanoid/game.py` (Actualizado)**

La clase principal del juego ahora creará y mantendrá la instancia del `AudioManager`.

```python
# arkanoid/game.py
import pygame
import os
from engine.asset_manager import AssetManager
from engine.audio import AudioManager # Importamos el gestor de audio
from .scenes.menu_scene import MenuScene
from .scenes.game_scene import GameScene

class ArkanoidGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((800, 600))
        pygame.display.set_caption("Arkanoid Profesional")
        self.clock = pygame.time.Clock()
        self.running = True
        
        self.game_dir = os.path.dirname(__file__)
        self.assets_dir = os.path.join(self.game_dir, 'assets') # Apuntamos a 'assets' en general
        
        # --- CAMBIO CLAVE: Creamos instancias de los gestores ---
        self.assets = AssetManager()
        self.audio_manager = AudioManager()
        
        self.load_assets()
        
        if self.running:
            self.setup_scenes()

    def load_assets(self):
        """Ahora también carga los sonidos a través del AudioManager."""
        try:
            sprites_path = os.path.join(self.assets_dir, 'sprites')
            audio_path = os.path.join(self.assets_dir, 'audio')
            
            # Carga de imágenes y spritesheets
            self.assets.load_spritesheet('main_ss', os.path.join(sprites_path, 'arkanoid.json'))
            self.assets.load_image('ball_img', os.path.join(sprites_path, 'ball.png'))

            # --- CAMBIO CLAVE: Carga de sonidos ---
            self.audio_manager.load_sound('hit_paddle', os.path.join(audio_path, 'hit_paddle.wav'))
            self.audio_manager.load_sound('hit_brick', os.path.join(audio_path, 'hit_brick.wav'))
            self.audio_manager.load_sound('lose_life', os.path.join(audio_path, 'lose_life.wav'))
            self.audio_manager.load_music(os.path.join(audio_path, 'music.ogg'))

        except Exception as e:
            print(f"Error fatal al cargar assets: {e}")
            self.running = False
            
    # ... (el resto de la clase game.py se mantiene igual)
```

#### **`arkanoid/scenes/game_scene.py` (Actualizado)**

Finalmente, la escena del juego simplemente llama a los métodos del gestor en los momentos adecuados.

```python
# arkanoid/scenes/game_scene.py
# ... (imports) ...

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        # ... (fuentes y colores)
        self.setup_game()

    def setup_game(self):
        # ... (código de setup_game sin cambios) ...
        self.estado_juego = 'jugando'
        self.game.audio_manager.play_music() # Inicia la música al empezar un nuevo juego

    def update(self):
        if self.estado_juego != 'jugando':
            return
        
        self.todos_los_sprites.update(self.game.screen.get_width(), self.game.screen.get_height())
        
        for bola in self.pelotas:
            # Colisión con la pala
            if bola.rect.colliderect(self.pala.rect):
                bola.velocidad_y *= -1
                bola.rect.bottom = self.pala.rect.top
                # --- CAMBIO CLAVE: Reproducir sonido ---
                self.game.audio_manager.play_sound('hit_paddle')

            # Colisión con los ladrillos
            ladrillos_golpeados = pygame.sprite.spritecollide(bola, self.ladrillos, False)
            if ladrillos_golpeados:
                ladrillo = ladrillos_golpeados[0]
                bola.velocidad_y *= -1
                if ladrillo.hit():
                    ladrillo.kill()
                    self.puntuacion += ladrillo.value
                    # --- CAMBIO CLAVE: Reproducir sonido ---
                    self.game.audio_manager.play_sound('hit_brick')
                    # ... (lógica de power-up) ...

        # ... (colisión con power-ups) ...
        
        # --- Lógica de Vidas ---
        if not self.pelotas:
            self.vidas -= 1
            if self.vidas > 0:
                self.spawn_ball()
                # --- CAMBIO CLAVE: Reproducir sonido ---
                self.game.audio_manager.play_sound('lose_life')
            else:
                self.estado_juego = 'game_over'
                # --- CAMBIO CLAVE: Parar la música ---
                self.game.audio_manager.stop_music()
        
        if not self.ladrillos:
            self.estado_juego = 'victoria_final'
            self.game.audio_manager.stop_music()

     #Ubica la pelota en la pala luego de perder.
     def spawn_ball(self):
        try:
            bola_img = self.game.assets.get_sprite('main_ss', 'pelota_bocha')
            nueva_bola = Ball(self.pala.rect.centerx, self.pala.rect.top - 20, bola_img)
            self.pelotas.add(nueva_bola)
            self.todos_los_sprites.add(nueva_bola)
        except (ValueError, KeyError) as e:
            print(f"Error al generar una nueva bola: {e}")

    # ... (el resto de la clase se mantiene igual)
```

-----

### **Resumen de la Actualización**

1.  **Motor de Audio (`engine/audio.py`):** Hemos añadido un `AudioManager` profesional al motor. Ahora tienes un lugar central para manejar todos los aspectos del sonido.
2.  **Gestor Principal (`arkanoid/game.py`):** La clase `ArkanoidGame` ahora crea una instancia del `AudioManager` y le dice qué sonidos y música cargar al inicio, centralizando la carga de todos los assets.
3.  **Escena de Juego (`arkanoid/scenes/game_scene.py`):** La escena del juego ya no se preocupa por CÓMO se reproduce el sonido. Simplemente le dice al gestor CUÁNDO hacerlo, llamando a `self.game.audio_manager.play_sound('nombre_del_sonido')`.

